# M5 Global EDA Notes

I am using this notebook to decide what is actually worth testing, not just to summarize the M5/Walmart subset. The main thing I care about is whether the model would learn true demand behavior, or whether the dataset has retail artifacts I need to handle first: item activation/ranging, sparse demand, calendar timing, store/state differences, and price movement.

The notebook delegates reproducible artifact generation to `backend/scripts/eda_m5_global.py`, then uses the exported JSON/Markdown/charts as the audit trail. I am intentionally leaving notes in here so the hypotheses do not look like they appeared out of nowhere.

In [ ]:
from pathlib import Path
import json
import subprocess

import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 40)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'backend').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data/benchmarks/m5_walmart/subset_20spc'
OUTPUT_DIR = ROOT / 'backend/reports/experiments/manual_ds/00_m5_global_eda'
REPORT_JSON = OUTPUT_DIR / 'm5_global_eda.json'
REPORT_MD = OUTPUT_DIR / 'm5_global_eda.md'
RAW_PATH = DATA_DIR / 'canonical_transactions.csv'
ROOT

## Generate Reproducible Artifacts

Run this whenever the M5 subset or EDA script changes. The output is what gets referenced from the manual DS story and can later be attached to a ShelfOps experiment record.

In [ ]:
subprocess.run(
    [
        'python3',
        str(ROOT / 'backend/scripts/eda_m5_global.py'),
        '--data-dir',
        str(DATA_DIR),
        '--output-dir',
        str(OUTPUT_DIR),
    ],
    cwd=ROOT,
    check=True,
)

In [ ]:
report = json.loads(REPORT_JSON.read_text())
summary = report['summary']
quality = report['data_quality_summary']
lifecycle = quality['lifecycle_summary']
late_launch = report['late_launch_deep_dive']
regionality = report['regionality_summary']
calendar = report['calendar_summary']
price = report['price_summary']
category = pd.DataFrame(report['category_summary'])
velocity = pd.DataFrame(report['velocity_summary'])
taxonomy = pd.DataFrame(report['intermittent_demand_summary']['taxonomy'])
summary

## Claim Boundary First

Before modeling, I want the data contract clear. M5 gives daily sales, hierarchy, sell price, events, and SNAP flags. It does not give true on-hand inventory, supplier lead times, purchase orders, shelf capacity, shrink, spoilage, or buyer decisions. So this EDA can support benchmark model and simulated decision hypotheses, but it cannot support measured merchant-impact claims.

In [ ]:
display(pd.DataFrame([{
    'rows': summary['rows'],
    'series': summary['series'],
    'days': summary['days'],
    'zero_sales_rate': summary['zero_sales_rate'],
    'price_coverage': summary['price_coverage'],
    'explicit_promo_row_rate': summary['explicit_promo_row_rate'],
    'event_day_row_rate': summary['event_day_row_rate'],
    'snap_row_rate': summary['snap_row_rate'],
}]))

display(pd.DataFrame(quality['column_profile']))
display(pd.DataFrame(quality['price_coverage_by_year']))
quality['retail_data_cautions']

Note to self: the grid is dense and duplicate-free, which is good for time-series evaluation. The danger is semantic, not structural. A zero row can mean true low demand, no shelf availability, or a SKU that was not effectively ranged yet. I should not let a model or policy treat every zero as equally informative.

## Demand Shape And Volume Concentration

Retail models live or die by segment behavior. If a small number of fast movers carry most unit volume, global metrics will reward those items and may hide slow-SKU overstock risk. If sparse series dominate count, replenishment policy needs different guardrails than the forecasting model.

In [ ]:
display(category[[
    'category',
    'products',
    'series',
    'total_units',
    'unit_share',
    'zero_sales_rate',
    'avg_series_nonzero_rate',
    'price_change_series_rate',
]])

display(velocity[[
    'velocity_segment',
    'series',
    'products',
    'total_units',
    'unit_share',
    'avg_daily_units',
    'avg_nonzero_rate',
    'price_change_series_rate',
]])

display(Markdown(f"Top 10% of series carry **{summary['top_10pct_series_unit_share']:.1%}** of units."))

Note to self: FOODS carries the most units and has the lowest zero-sales rate, but sparse behavior is still high. Fast series carry a majority of units, while intermittent plus slow series are a large share of the SKU-store count. This is exactly where a global model metric can look better while buyer-facing order decisions get worse for slow movers.

## Lifecycle / Ranging Risk

I need to separate steady-state zero demand from assortment timing before I start feature work. Item launches, delists, and dormant tails can poison both forecasting and replenishment logic.

In [ ]:
display(pd.DataFrame([{
    'series_with_no_sales_rate': lifecycle['series_with_no_sales_rate'],
    'late_launch_series_rate': lifecycle['late_launch_series_rate'],
    'dormant_tail_series_rate': lifecycle['dormant_tail_series_rate'],
    'pre_first_sale_zero_row_share': lifecycle['pre_first_sale_zero_row_share'],
    'post_last_sale_zero_row_share': lifecycle['post_last_sale_zero_row_share'],
    'avg_active_window_zero_rate': lifecycle['avg_active_window_zero_rate'],
}]))

display(pd.DataFrame(lifecycle['by_category'])[[
    'category',
    'series',
    'late_launch_rate',
    'dormant_tail_rate',
    'median_first_sale_offset_days',
    'median_last_sale_tail_days',
    'avg_active_window_zero_rate',
]])

Note to self: this is a real modeling warning. Late-launch behavior is common enough that a naive full-history model can learn too much from pre-launch zeros. This points toward an activation-aware data spec before I reach for a new architecture.

### Late-Launch SKU Drilldown: Seasonal Or Ranging?

This is the follow-up investigation. A late first sale could be true seasonality, but it could also be item introduction, store ranging, missing availability, or a sparse new product. I am checking this at SKU-store level: first sale vs. first available sell price, whether the SKU keeps selling near the end of the data, and whether units concentrate in a narrow recurring month window.

In [ ]:
display(pd.DataFrame([{
    'late_launch_series': late_launch['late_launch_series'],
    'price_aligned_late_launch_rate': late_launch['price_aligned_late_launch_rate'],
    'likely_late_ranged_or_introduced_rate': late_launch['likely_late_ranged_or_introduced_rate'],
    'possible_seasonal_rate': late_launch['possible_seasonal_rate'],
}]))

display(pd.DataFrame(late_launch['classification_summary'])[[
    'classification',
    'series',
    'products',
    'total_units',
    'unit_share_within_late_launch',
    'avg_first_sale_offset_days',
    'avg_active_span_days',
    'avg_positive_months',
    'avg_top_3_month_unit_share',
    'price_aligned_rate',
]])

display(pd.DataFrame(late_launch['by_category']))

Note to self: this does **not** look like classic seasonality in the late-launch cohort. The strongest signal is price/ranging alignment: nearly every late-launch series has sell price appear within roughly two weeks of the first sale. Most of those SKUs then sell across many months and continue near the dataset end. That points to item/store introduction or ranging, not a recurring seasonal-only item.

In [ ]:
display(Markdown('#### High-volume late-launch examples'))
display(pd.DataFrame(late_launch['high_volume_examples'])[[
    'store_id',
    'product_id',
    'category',
    'first_sale_date',
    'first_price_date',
    'first_price_minus_first_sale_days',
    'last_sale_date',
    'total_units',
    'positive_months',
    'top_months_by_units',
    'top_3_month_unit_share',
    'classification',
]])

display(Markdown('#### Most delayed late-launch examples'))
display(pd.DataFrame(late_launch['most_delayed_examples'])[[
    'store_id',
    'product_id',
    'category',
    'first_sale_date',
    'first_price_date',
    'last_sale_date',
    'total_units',
    'nonzero_days',
    'positive_months',
    'top_3_month_unit_share',
    'classification',
]])

Note to self: the examples matter because they prevent me from overgeneralizing. Some very delayed SKUs are too sparse to classify confidently. I should handle those conservatively in experiments, probably as a lifecycle/sparse segment rather than as evidence for seasonal modeling.

## Intermittent Demand Taxonomy

The velocity segments are operationally useful, but I also want a demand-theory view. I am using a Syntetos-Boylan style classification with ADI and squared CV on nonzero demand. This helps decide whether a SKU needs forecast tuning, a different objective, or a different ordering policy.

In [ ]:
display(taxonomy[[
    'demand_classification',
    'series',
    'products',
    'total_units',
    'unit_share',
    'avg_nonzero_rate',
    'median_adi',
    'median_cv2',
    'median_intersale_gap_days',
    'p90_intersale_gap_days',
]])

taxonomy_by_category = pd.DataFrame(report['intermittent_demand_summary']['by_category'])
display(taxonomy_by_category.pivot_table(
    index='category',
    columns='demand_classification',
    values='series',
    aggfunc='sum',
    fill_value=0,
))
display(taxonomy_by_category.pivot_table(
    index='category',
    columns='demand_classification',
    values='unit_share_within_category',
    aggfunc='sum',
    fill_value=0,
))

Note to self: this is bigger than a forecast feature problem. A lot of the benchmark is intermittent/lumpy by item-store behavior. That means I should evaluate model metrics and decision metrics together: WAPE/MASE/bias plus simulated overstock, stockout opportunity cost, and service-level behavior.

## Calendar, SNAP, And Event Windows

I do not want to throw holiday flags into a model and call it domain knowledge. The retail question is whether timing effects differ by state and category enough to justify interactions.

In [ ]:
display(pd.DataFrame(calendar['weekday']))
display(pd.DataFrame(calendar['snap']))
display(pd.DataFrame(calendar['snap_by_category']))
display(pd.DataFrame(calendar['event_type_effects']))
display(pd.DataFrame(calendar['event_name_effects']).head(12))

Note to self: SNAP appears meaningfully stronger for FOODS, especially in WI and TX in this subset. Weekends are also higher volume. Calendar features should be tested with state/category interactions, but after the activation-aware dataset test. I do not want to mix a data-contract change and a calendar-feature change in the same first experiment.

## Price And Promo Proxy

M5 has sell prices, but this canonical subset does not carry explicit promotion labels. Price movement could still be useful, but I need to be careful with claims. Price drops are a proxy hypothesis, not proof of ad or promotion response.

In [ ]:
display(pd.DataFrame([{
    'price_coverage': price['price_coverage'],
    'series_with_price_changes_rate': price['series_with_price_changes_rate'],
    'rows_with_price_change_rate': price['rows_with_price_change_rate'],
    'rows_with_price_drop_rate': price['rows_with_price_drop_rate'],
    'median_series_price_quantity_correlation': price['median_series_price_quantity_correlation'],
}]))
display(pd.DataFrame(price['category_price_effects'])[[
    'category',
    'price_change_row_rate',
    'price_drop_row_rate',
    'price_drop_uplift_vs_no_change',
]])
display(pd.DataFrame(price['category_price_correlations']))
display(pd.DataFrame(price['top_price_mover_series']))

Note to self: price features are plausible, but not my first move. Many series have price movement, but price-change rows are sparse, explicit promotion is unavailable, and category-level response is mixed. I would queue this after activation/calendar/intermittent-demand work unless model diagnostics show price-window failures.

## Regionality / Store Heterogeneity Drilldown

This is the next thing I am seeing. Regionality is not just a state label. It can show up as store volume scale, category mix, SNAP sensitivity, weekend timing, and different intermittent/lumpy demand shares. The question is whether this deserves a logged feature hypothesis or is just descriptive noise.

In [ ]:
display(pd.DataFrame([regionality['reads']]))
display(pd.DataFrame(regionality['state_summary'])[[
    'state_id',
    'stores',
    'products',
    'total_units',
    'unit_share',
    'avg_units_per_row',
    'zero_sales_rate',
    'price_coverage',
]])
display(pd.DataFrame(regionality['store_summary'])[[
    'state_id',
    'store_id',
    'total_units',
    'unit_share',
    'avg_units_per_row',
    'zero_sales_rate',
    'price_coverage',
]])

In [ ]:
display(Markdown('#### Category x state demand shape'))
display(pd.DataFrame(regionality['category_state_summary'])[[
    'category',
    'state_id',
    'products',
    'total_units',
    'unit_share_within_category',
    'avg_units_per_row',
    'zero_sales_rate',
    'index_vs_category_avg',
]])

display(Markdown('#### Weekend uplift by state/category'))
display(pd.DataFrame(regionality['weekend_by_category_state'])[[
    'state_id',
    'category',
    'weekday_avg_units',
    'weekend_avg_units',
    'weekend_uplift',
]])

display(Markdown('#### Late activation by state/category'))
display(pd.DataFrame(regionality['late_launch_by_category_state'])[[
    'category',
    'state_id',
    'series',
    'late_series',
    'late_rate',
    'total_units',
    'late_units',
    'late_unit_share',
]])

Note to self: regionality is strong enough to become a hypothesis, but it needs to be scoped. The best candidate is not simply `add state`. The evidence points to **state/category calendar interactions**: FOODS has materially different SNAP response by state, weekend uplift differs by category and state, and store volume scale varies by more than 2x. This becomes a clean second experiment after activation-aware data construction.

## What Else I Am Seeing

At this point the EDA is pointing to a sequence, not one magic feature:

1. **Activation semantics:** pre-price/pre-first-sale zeros are likely not active-demand observations. This is the cleanest data-contract hypothesis.
2. **Regional timing:** state/category interactions matter, especially FOODS SNAP response and weekend uplift. This is a feature hypothesis.
3. **Intermittent/lumpy behavior:** a large share of unit volume is intermittent or lumpy. This is probably a model-policy hypothesis, not just a feature tweak.
4. **Velocity concentration:** fast movers carry most units, but sparse items carry operational risk. This means global metrics can mislead us.
5. **Price/promo proxy:** price movement exists, but explicit promo is missing and price-change rows are sparse. This is real but should come later.

The important discipline: each hypothesis should change one layer at a time: dataset spec, then features, then objective/policy, then model architecture if needed.

## Hypothesis Register

This is my handoff from EDA to experiment design. Each row links an observed finding to a retail interpretation, a concrete experiment change, the metrics I expect to move, the risk, and confidence level. These are not results yet; they are testable hypotheses.

In [ ]:
hypothesis_register = pd.DataFrame(report['hypothesis_register'])
display(hypothesis_register[[
    'rank',
    'finding',
    'evidence',
    'domain_interpretation',
    'hypothesis',
    'experiment_change',
    'primary_metrics',
    'risk',
    'confidence',
]])

## Evaluation Plan Before Experiments

Before running anything through ShelfOps, I need to define what would count as useful evidence. The goal is not to make every metric better everywhere; it is to improve the target failure mode while keeping guardrails stable. All decision metrics from M5 stay simulated/provisional because M5 does not include live inventory, POs, or buyer decisions.

In [ ]:
evaluation_plan = pd.DataFrame(report['evaluation_plan'])
display(evaluation_plan[[
    'rank',
    'hypothesis',
    'experiment_artifact',
    'primary_slice',
    'primary_success_metric',
    'secondary_metrics',
    'guardrail',
    'pass_condition',
]])

Note to self: this register is what makes the next notebooks defensible. If a hypothesis does not have a finding, metric, and guardrail here, it should not get promoted into a full experiment yet.

## Candidate Manual DS Hypotheses

These are the hypotheses I would log in ShelfOps from this first EDA. They are ranked by expected learning value, not by complexity.

In [ ]:
for hypothesis in report['candidate_hypotheses']:
    display(Markdown(f"### {hypothesis['rank']}. {hypothesis['title']}"))
    display(Markdown(f"**Type:** `{hypothesis['experiment_type']}`"))
    display(Markdown(hypothesis['rationale']))
    display(Markdown('**Evidence**'))
    display(pd.DataFrame([hypothesis['evidence']]))
    display(Markdown('**Expected metric movement**'))
    display(pd.DataFrame([hypothesis['expected_metric_movement']]))

## Chart Artifacts

In [ ]:
for chart in report.get('charts', []):
    display(Markdown(f"### `{Path(chart).name}`"))
    display(Image(filename=str(ROOT / chart)))

## DS Decision

First manual experiment candidate: **activation-aware training window for pre-sellable SKU-store rows**.

Why this first: the EDA found a semantic data issue before a model issue. Many late-start SKU-store rows look like pre-activation/ranging periods, not true no-demand. If we train on those as normal zeros, the model can learn fake zero demand before an item was commercially active.

What I would log into ShelfOps next: a reproducible activation-aware dataset spec. Baseline keeps full canonical M5. Challenger excludes or downweights rows before first available sell price per SKU-store, with metrics reported globally and by late-launch, non-late, category, state, velocity, and intermittent/lumpy segments.

## Manual DS Track Status After Activation Runs

The activation/ranging issue from this EDA became the first manual DS hypothesis family.

What happened:

| experiment | idea | senior read |
|---|---|---|
| H01 | remove all pre-activation rows | rejected; slow/medium improved, but aggregate error and overstock worsened |
| H02 | remove pre-activation rows only for eligible slow/medium/late-activation series | better than H01, but bias/coverage/overstock still failed |
| H03 | route activation forecasts only to eligible series and keep champion forecasts for protected fast movers | cleanest model result, but lost-sales and stockout risk worsened |

Decision: activation is a real segment signal, but it is no longer the main hypothesis path. It should remain documented as a future segment/policy feature. The next hypothesis should return to EDA and test a new demand driver.

Next candidate families from this notebook:

1. Price/promotion proxy features from sell-price movement.
2. Horizon or segment-specific calibration, because business cost moved differently from WAPE.
3. Intermittent-demand treatment for slow/medium demand patterns.
